### Business Context

Renewable energy sources play an increasingly important role in the global energy mix, as the effort to reduce the environmental impact of energy production increases.

Out of all the renewable energy alternatives, wind energy is one of the most developed technologies worldwide. The U.S. Department of Energy has put together a guide to achieving operational efficiency using predictive maintenance practices.

Predictive maintenance uses sensor information and analysis methods to measure and predict degradation and future component capability. The idea behind predictive maintenance is that failure patterns are predictable and if component failure can be predicted accurately and the component is replaced before it fails, the costs of operation and maintenance will be much lower.

The sensors fitted across different machines involved in the process of energy generation collect data related to various environmental factors (temperature, humidity, wind speed, etc.) and additional features related to various parts of the wind turbine (gearbox, tower, blades, break, etc.).

### Objective

“ReneWind” is a company working on improving the machinery/processes involved in the production of wind energy using machine learning and has collected data on generator failure of wind turbines using sensors. They have shared a ciphered version of the data, as the data collected through sensors is confidential (the type of data collected varies with companies). Data has 40 predictors, 20000 observations in the training set, and 5000 in the test set.

The objective is to build various classification models, tune them, and find the best one that will help identify failures so that the generators can be repaired before failing/breaking to reduce the overall maintenance cost.
The nature of predictions made by the classification model will translate as follows:

True positives (TP) are failures correctly predicted by the model. These will result in repair costs.
False negatives (FN) are real failures where there is no detection by the model. These will result in replacement costs.
False positives (FP) are detections where there is no failure. These will result in inspection costs.
It is given that the cost of repairing a generator is much less than the cost of replacing it, and the cost of inspection is less than the cost of repair.

“1” in the target variable should be considered as “failure” and “0” represents “No failure”.

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix, classification_report, recall_score
from sklearn.utils.class_weight import compute_class_weight
import time  # Module for time-related operations.
from keras import backend   # Abstraction layer for neural network backend engines.
from keras.models import Sequential  # Model for building NN sequentially.
from keras.layers import Dense,Dropout,BatchNormalization, Input   # for creating fully connected neural network layers.


In [82]:
#Loading data
df = pd.read_csv("Y:/AIML Course/Neural Network/NN Project/Train.csv")
# --- detect target column (simple heuristic)
def detect_target(train_df):
    for name in ["Target"]:
        if name in df.columns:
            return name
    return df.columns[-1]

target = detect_target(df)


In [83]:
# --- Data overview
print("Overview")
print("Shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes)
print("\nNumber of missing values per column (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nNumber of unique values per column (top 10):")
print(df.nunique().sort_values(ascending=False).head(10))
print(f"\nDetected target column: '{target}'")
print("Target value counts:")
print(df[target].value_counts())


Overview
Shape: (20000, 41)

Columns and dtypes:
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
V29       float64
V30       float64
V31       float64
V32       float64
V33       float64
V34       float64
V35       float64
V36       float64
V37       float64
V38       float64
V39       float64
V40       float64
Target      int64
dtype: object

Number of missing values per column (top 10):
V1     18
V2     18
V3      0
V4      0
V5      0
V6      0
V7      0
V8      0
V9      0
V10     0
dtype: int64

Number of unique values per column (to

The dataset contains 20,000 observations with 40 numerical sensor features and one target column. Only two variables (V1 and V2) have a very small number of missing values. Most features contain fully unique values. The target variable is highly imbalanced, with failures representing only about 5–6% of the data.

In [84]:
# --- Univariate analysis
print("\n\n=== UNIVARIATE ANALYSIS ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric features: {len(numeric_cols)}")

# Basic stats for numeric features
print("\nNumeric features summary (first 10 rows):")
print(df[numeric_cols].describe().T.head(10))

# Plot histograms for first up to 12 numeric features
plot_cols = numeric_cols[:12]
n = len(plot_cols)
if n > 0:
    cols = 3
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(4*cols, 3*rows))
    for i, c in enumerate(plot_cols, 1):
        plt.subplot(rows, cols, i)
        sns.histplot(df[c].dropna(), bins=40, kde=True)
        plt.title(c)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns to plot.")


=== UNIVARIATE ANALYSIS ===
Numeric features: 41

Numeric features summary (first 10 rows):
       count      mean       std        min       25%       50%       75%  \
V1   19982.0 -0.271996  3.441625 -11.876451 -2.737146 -0.747917  1.840112   
V2   19982.0  0.440430  3.150784 -12.319951 -1.640674  0.471536  2.543967   
V3   20000.0  2.484699  3.388963 -10.708139  0.206860  2.255786  4.566165   
V4   20000.0 -0.083152  3.431595 -15.082052 -2.347660 -0.135241  2.130615   
V5   20000.0 -0.053752  2.104801  -8.603361 -1.535607 -0.101952  1.340480   
V6   20000.0 -0.995443  2.040970 -10.227147 -2.347238 -1.000515  0.380330   
V7   20000.0 -0.879325  1.761626  -7.949681 -2.030926 -0.917179  0.223695   
V8   20000.0 -0.548195  3.295756 -15.657561 -2.642665 -0.389085  1.722965   
V9   20000.0 -0.016808  2.160568  -8.596313 -1.494973 -0.067597  1.409203   
V10  20000.0 -0.012998  2.193201  -9.853957 -1.411212  0.100973  1.477045   

           max  
V1   15.493002  
V2   13.089269  
V3   17.0

The univariate summary shows that all numeric features exhibit a wide range of values, reflecting the natural variability in turbine sensor measurements. Most variables have means close to zero and relatively symmetric distributions, while some display larger spreads, indicating higher volatility in certain sensor readings. The presence of both negative and positive values suggests normalized or transformed sensor data. Overall, the numerical features show sufficient variation and no obvious anomalies, making them suitable for downstream modeling.

In [85]:
# Plot boxplots for first 12 numeric features
plot_cols = numeric_cols[:12]  

n = len(plot_cols)
cols = 3
rows = (n + cols - 1) // cols

plt.figure(figsize=(5*cols, 4*rows))
for i, feature in enumerate(plot_cols, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(x=df[feature])
    plt.title(feature)

plt.tight_layout()
plt.show()


The boxplots show that most sensor features contain several outliers on both ends of the distribution, which is common in real-world turbine monitoring data due to sudden environmental or mechanical fluctuations. Despite the presence of these extreme values, the central boxes are narrow and consistent, indicating stable median behavior and relatively low variability for the majority of observations.

## Bivariate Analysis

In [86]:
# Correlation of numeric features with target (convert target to numeric if needed)
if target in numeric_cols:
    corr_with_target = df[numeric_cols].corr()[target]
else:
    # convert target to numeric for correlation (0/1 expected)
    corr_with_target = df[numeric_cols].corrwith(pd.to_numeric(df[target], errors="coerce"))

# Show absolute correlations sorted
corr_abs = corr_with_target.abs().sort_values(ascending=False)
print("\nTop 10 features correlated with target (absolute correlation):")
print(corr_abs.head(10))


Top 10 features correlated with target (absolute correlation):
Target    1.000000
V18       0.293340
V21       0.256411
V15       0.249118
V7        0.236907
V16       0.230507
V39       0.227264
V36       0.216453
V3        0.213855
V28       0.207359
Name: Target, dtype: float64

The correlation analysis shows that several sensor features—including V18, V21, V15, V7, and V16—have the strongest relationships with the failure target, though the correlations remain moderate. This suggests that turbine failures are influenced by a combination of factors rather than a single dominant variable. These top features may reflect key mechanical or environmental stress indicators, making them valuable inputs for predictive maintenance models.

In [87]:
# Scatter / boxplots for top 6 features vs target
top_feats = corr_abs.head(6).index.tolist()
print("\nPlotting top features vs target (scatter or boxplot depending on target type)...")
numeric_target = pd.api.types.is_numeric_dtype(df[target])
n = len(top_feats)
cols = 2
rows = (n + cols - 1) // cols
plt.figure(figsize=(6*cols, 4*rows))
for i, f in enumerate(top_feats, 1):
    plt.subplot(rows, cols, i)
    if numeric_target:
        sns.scatterplot(x=df[f], y=df[target], alpha=0.4, s=10)
        plt.ylabel(target)
    else:
        # boxplot grouped by target categories
        sns.boxplot(x=df[target].astype(str), y=df[f])
        plt.xlabel(target)
        plt.ylabel(f)
    plt.title(f"{f} vs {target}")
plt.tight_layout()
plt.show()


Plotting top features vs target (scatter or boxplot depending on target type)...

The scatterplots between the most correlated features and the target show clear separation patterns, with failure cases (Target = 1) appearing in distinct value ranges compared to non-failures. Although the target is binary and forms horizontal bands, features such as V18, V21, V15, V7, and V16 display noticeable clustering that differentiates failing from healthy turbines. This indicates that these sensor readings capture meaningful behavioral changes linked to generator failure and are therefore important contributors to the predictive model.

In [88]:
# Correlation heatmap for top numeric features
top_corr_feats = corr_abs.head(12).index.tolist()
if len(top_corr_feats) >= 2:
    plt.figure(figsize=(10,8))
    sns.heatmap(df[top_corr_feats].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Correlation matrix (top features)")
    plt.show()
else:
    print("Not enough features for heatmap.")


The correlation heatmap highlights that several top features show moderate relationships with the failure target and also exhibit interdependencies among themselves. Features such as V18, V21, V15, and V16 form a cluster of positively correlated variables, suggesting they may capture related mechanical or environmental stress patterns. Meanwhile, some features like V3 and V26 show negative correlations with others, indicating contrasting sensor behavior. Overall, the heatmap points that turbine failures are influenced by multiple interacting factors rather than a single dominant variable.

In [89]:
# Take first 8 numeric features for plotting
biv_cols = numeric_cols[:8]   # numeric_cols already defined earlier

plt.figure(figsize=(14, 12))

for i, col in enumerate(biv_cols, 1):
    plt.subplot(4, 2, i)
    sns.violinplot(x=df[target], y=df[col])
    plt.title(f"{col} vs Target")
    plt.xlabel("Target (0 = No Failure, 1 = Failure)")
    plt.ylabel(col)

plt.tight_layout()
plt.show()


The violin plots show that the distributions of most sensor readings are broadly similar for both failure and non-failure cases, but with subtle shifts in their central tendency and spread. These slight distributional differences indicate that certain sensor values occur more frequently in turbines that eventually fail, even though the overall ranges overlap.

In [90]:
#Missing Value Treatment

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = [c for c in df.columns if c not in numeric_cols]

print("Number of numeric columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

# 1. Numeric columns → replace missing values with median
for col in numeric_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# 2. Categorical columns → replace missing with most frequent value (mode)
for col in categorical_cols:
    mode_value = df[col].mode()[0]   # mode()[0] gives the most frequent value
    df.method({col: value}, inplace=True)

print("\nMissing value treatment completed!")

# Verify no missing values remain
print("\nRemaining missing values in each column:")
print(df.isnull().sum())


Number of numeric columns: 41
Number of categorical columns: 0

Missing value treatment completed!

Remaining missing values in each column:
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
V29       0
V30       0
V31       0
V32       0
V33       0
V34       0
V35       0
V36       0
V37       0
V38       0
V39       0
V40       0
Target    0
dtype: int64

Missing value treatment done and now we have 0 missing values in the coloumns

In [91]:
test_df = pd.read_csv("Y:/AIML Course/Neural Network/NN Project/Test.csv")  # test data

# Separate X and y from df
X = df.drop(columns=[target])
y = df[target].astype(int)

# Separate X and y from test_df
X_test = test_df.drop(columns=[target])
y_test = test_df[target].astype(int)

# Train–Validation Split (stratified)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape (separate):", X_test.shape)


Train shape: (16000, 40)
Validation shape: (4000, 40)
Test shape (separate): (5000, 40)

In [92]:
def make_tf_dataset(X_np, y_series=None, batch_size=64, shuffle=False):
    if y_series is not None:
        y_np = y_series.to_numpy().astype("float32")
        ds = tf.data.Dataset.from_tensor_slices((X_np.astype("float32"), y_np))
    else:
        ds = tf.data.Dataset.from_tensor_slices((X_np.astype("float32")))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X_np), seed=42)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(X_train, y_train, shuffle=True)
val_ds   = make_tf_dataset(X_val, y_val)
test_ds  = make_tf_dataset(X_test)

# Quick batch check
for xb, yb in train_ds.take(1):
    print("Batch X:", xb.shape, "Batch y:", yb.shape)


Batch X: (64, 40) Batch y: (64,)

In [93]:
def plot(history, name):
    """
    Function to plot loss/accuracy

    history: an object which stores the metrics and losses.
    name: can be one of Loss or Accuracy
    """
    fig, ax = plt.subplots() #Creating a subplot with figure and axes.
    plt.plot(history.history[name]) #Plotting the train accuracy or train loss
    plt.plot(history.history['val_'+name]) #Plotting the validation accuracy or validation loss

    plt.title('Model ' + name.capitalize()) #Defining the title of the plot.
    plt.ylabel(name.capitalize()) #Capitalizing the first letter.
    plt.xlabel('Epoch') #Defining the label for the x-axis.
    fig.legend(['Train', 'Validation'], loc="outside right upper") #Defining the legend, loc controls the position of the legend.


In [94]:
cols = ["id","hidden","activations","epochs","batch_size","optimizer","lrs","initializer",
        "notes","train_loss","val_loss","train_acc","val_acc","train_time"]

results = pd.DataFrame(columns=cols, dtype=object)


In [95]:
#Model Building

batch_size=32
epochs=50
num_classes=10


In [96]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()


## Model 0

In [97]:
#Initializing the neural network
model_0 = Sequential()
model_0.add(Input(shape=(X_train.shape[1],)))
model_0.add(Dense(128, activation="relu"))
model_0.add(Dense(64, activation="tanh"))
model_0.add(Dense(num_classes, activation="softmax"))


In [98]:
model_0.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
10
)                  │             
650
 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

Total params: 
14,154
 (55.29 KB)

Trainable params: 
14,154
 (55.29 KB)

Non-trainable params: 
0
 (0.00 B)

In [99]:
optimizer = keras.optimizers.SGD()    # defining SGD as the optimizer to be used
model_0.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [100]:
start = time.time()
history = model_0.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()
print("Training time (s):", end - start)


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9551 - loss: 0.1773 - val_accuracy: 0.9725 - val_loss: 0.0952
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9776 - loss: 0.0837 - val_accuracy: 0.9822 - val_loss: 0.0709
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9837 - loss: 0.0681 - val_accuracy: 0.9855 - val_loss: 0.0602
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9858 - loss: 0.0599 - val_accuracy: 0.9885 - val_loss: 0.0554
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9879 - loss: 0.0551 - val_accuracy: 0.9900 - val_loss: 0.0524
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9894 - loss: 0.0519 - val_accuracy: 0.9890 - val_loss: 0.0504
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9899 - loss: 0.0493 - val_accuracy: 0.9908 - val_loss: 0.0487
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - ac

In [101]:
plot(history,'loss')
plot(history,'accuracy')


In [102]:
train_loss = history.history.get("loss", [None])[-1]
val_loss   = history.history.get("val_loss", [None])[-1]
train_acc  = history.history.get("accuracy", history.history.get("acc", [None]))[-1]
val_acc    = history.history.get("val_accuracy", history.history.get("val_acc", [None]))[-1]
train_time = round(end - start, 2)

results.loc[len(results)] = [
    1,
    [128,64],
    ["relu","tanh"],
    50,
    32,
    "sgd",
    [0.001, "-"],
    "xavier",
    "-",
    train_loss,
    val_loss,
    train_acc,
    val_acc,
    train_time
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.03962


0.99425


0.9925


85.13

#### Model 1 — SGD

This baseline model performs strongly, achieving high training and validation accuracy with low loss. SGD learns steadily but not as adaptively as modern optimizers, yet it still delivers reliable convergence due to the simplicity of the architecture and dataset.

## Model 1

In [103]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()


In [104]:
#Initializing the neural network
model_1 = Sequential()
model_1.add(Dense(128,activation="relu",input_dim = X_train.shape[1]))
model_1.add(Dense(64,activation="tanh"))
model_1.add(Dense(num_classes,activation = 'softmax'))


C:\Users\yyyad\anaconda3\envs\notebook-6.5.7\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)

In [105]:
model_1.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
10
)                  │             
650
 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

Total params: 
14,154
 (55.29 KB)

Trainable params: 
14,154
 (55.29 KB)

Non-trainable params: 
0
 (0.00 B)

In [106]:
mom = 0.9    # defining the momentum value to be used
optimizer = keras.optimizers.SGD(momentum=mom)    # defining the SGD optimizer with a specific momentum

model_1.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [107]:
start = time.time()
history = model_1.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()
print("Training time (s):", end - start)


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9781 - loss: 0.0893 - val_accuracy: 0.9905 - val_loss: 0.0502
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9891 - loss: 0.0499 - val_accuracy: 0.9908 - val_loss: 0.0463
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9908 - loss: 0.0439 - val_accuracy: 0.9925 - val_loss: 0.0408
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9921 - loss: 0.0401 - val_accuracy: 0.9915 - val_loss: 0.0423
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9918 - loss: 0.0387 - val_accuracy: 0.9910 - val_loss: 0.0487
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9926 - loss: 0.0359 - val_accuracy: 0.9927 - val_loss: 0.0413
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9929 - loss: 0.0354 - val_accuracy: 0.9923 - val_loss: 0.0404
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - ac

In [108]:
print("Time taken in seconds ",end-start)


Time taken in seconds  109.01682806015015

In [109]:
plot(history,'loss')


In [110]:
plot(history,'accuracy')


In [111]:
train_loss = history.history.get("loss", [None])[-1]
val_loss   = history.history.get("val_loss", [None])[-1]
train_acc  = history.history.get("accuracy", history.history.get("acc", [None]))[-1]
val_acc    = history.history.get("val_accuracy", history.history.get("val_acc", [None]))[-1]
train_time = round(end - start, 2)

results.loc[len(results)] = [
    2,
    [128,64],
    ["relu","tanh"],
    50,
    32,
    "sgd with momentum",
    [0.001, mom],
    "xavier",
    "-",
    train_loss,
    val_loss,
    train_acc,
    val_acc,
    train_time
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.99425


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.99700


0.99125


109.02

#### Model 2 — SGD with Momentum

Momentum helps accelerate convergence and smooth out noisy gradient updates. The model achieves nearly identical accuracy to plain SGD, but benefits from more stable optimization dynamics. Still, it does not outperform adaptive optimizers like Adam.

## Model 2

In [112]:
# clears the current Keras session, resetting all layers and models previously created, freeing up memory and resources.
tf.keras.backend.clear_session()


In [113]:
#Initializing the neural network
model_2 = Sequential()
model_2.add(Dense(128,activation="relu",input_dim = X_train.shape[1]))
model_2.add(Dense(64,activation="tanh"))
model_2.add(Dense(num_classes,activation = 'softmax'))


C:\Users\yyyad\anaconda3\envs\notebook-6.5.7\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)

In [114]:
model_2.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
10
)                  │             
650
 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

Total params: 
14,154
 (55.29 KB)

Trainable params: 
14,154
 (55.29 KB)

Non-trainable params: 
0
 (0.00 B)

In [115]:
mom = 0.4
optimizer = keras.optimizers.SGD(momentum=mom)
model_2.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [116]:
start = time.time()
history = model_2.fit(X_train, y_train, validation_data=(X_val,y_val) , batch_size=batch_size, epochs=epochs)
end=time.time()
print("Training time (s):", end - start)


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9645 - loss: 0.1351 - val_accuracy: 0.9847 - val_loss: 0.0719
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9842 - loss: 0.0682 - val_accuracy: 0.9895 - val_loss: 0.0566
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9874 - loss: 0.0581 - val_accuracy: 0.9902 - val_loss: 0.0501
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9889 - loss: 0.0527 - val_accuracy: 0.9923 - val_loss: 0.0481
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9899 - loss: 0.0492 - val_accuracy: 0.9923 - val_loss: 0.0454
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9908 - loss: 0.0466 - val_accuracy: 0.9925 - val_loss: 0.0452
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 2ms/step - accuracy: 0.9908 - loss: 0.0448 - val_accuracy: 0.9930 - val_loss: 0.0426
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - ac

In [117]:
plot(history,'loss')


In [118]:
plot(history,'accuracy')


In [119]:
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end-start, 2)

results.loc[len(results)] = [
    3,
    [128,64],
    ["relu","tanh"],
    epochs,
    batch_size,
    "adam",
    [0.001, "-"],
    "glorot_uniform",
    "-",
    train_loss, val_loss, train_acc, val_acc, train_time
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52

#### Model 3 — Adam (Glorot Uniform)

Adam improves optimization stability by adapting learning rates during training. The model converges smoothly and achieves excellent accuracy with consistently low losses, performing slightly better in stability than the SGD variants.

## Model 3

In [120]:
# change initializer to he_normal, use Adam optimizer, hidden activations = elu
tf.keras.backend.clear_session()
model_3 = Sequential()
model_3.add(Input(shape=(X_train.shape[1],)))
model_3.add(Dense(128, activation="elu", kernel_initializer="he_normal"))
model_3.add(Dropout(0.2))
model_3.add(Dense(64, activation="elu", kernel_initializer="he_normal"))
# extra hidden layer added
model_3.add(Dense(32, activation="elu", kernel_initializer="he_normal"))
model_3.add(Dropout(0.2))
model_3.add(Dense(num_classes, activation="softmax"))
model_3.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (
Dropout
)                    │ (
None
, 
128
)                 │               
0
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
32
)                  │           
2,080
 │
├──────────────────────────────────────┼─────────────────────────────┼──

Total params: 
15,914
 (62.16 KB)

Trainable params: 
15,914
 (62.16 KB)

Non-trainable params: 
0
 (0.00 B)

In [121]:
optimizer = keras.optimizers.Adam(learning_rate=0.001)
model_3.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

start = time.time()
history = model_3.fit(X_train, y_train, validation_data=(X_val,y_val), batch_size=batch_size, epochs=epochs)
end = time.time()
print("Training time (s):", round(end - start,2))


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9405 - loss: 0.2847 - val_accuracy: 0.9860 - val_loss: 0.0655
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9797 - loss: 0.0856 - val_accuracy: 0.9908 - val_loss: 0.0498
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9846 - loss: 0.0685 - val_accuracy: 0.9895 - val_loss: 0.0522
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9868 - loss: 0.0602 - val_accuracy: 0.9912 - val_loss: 0.0454
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9878 - loss: 0.0553 - val_accuracy: 0.9918 - val_loss: 0.0424
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 2ms/step - accuracy: 0.9896 - loss: 0.0533 - val_accuracy: 0.9920 - val_loss: 0.0418
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9889 - loss: 0.0518 - val_accuracy: 0.9923 - val_loss: 0.0422
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - ac

In [122]:
plot(history,'loss')
plot(history,'accuracy')


In [123]:
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end - start, 2)

# append Model 3 entry to results
results.loc[len(results)] = [
    4,
    [128, 64, 32],
    ["elu", "elu", "elu"],
    epochs,
    batch_size,
    "adam",
    [0.001, "-"],
    "he_normal",
    "-",
    train_loss, val_loss, train_acc, val_acc, train_time
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52






3


4


[128, 64, 32]


[elu, elu, elu]


50


32


adam


[0.001, -]


he_normal


-


0.028050


0.054394


0.993813


0.99300


120.34

#### Model 4 — Adam (He Normal)

This model uses He initialization, which works well with ELU activations. Training is smooth and validation performance matches Model 3. The combination of Adam + He Normal offers strong convergence without requiring additional tuning.

## Model 4

In [124]:
# use Glorot uniform initializer, RMSprop optimizer, activation = selu
tf.keras.backend.clear_session()
model_4 = Sequential()
model_4.add(Input(shape=(X_train.shape[1],)))
model_4.add(Dense(256, activation="selu", kernel_initializer="glorot_uniform"))
model_4.add(Dropout(0.15))
model_4.add(Dense(128, activation="selu", kernel_initializer="glorot_uniform"))
model_4.add(Dense(64, activation="selu", kernel_initializer="glorot_uniform"))  # extra layer
model_4.add(Dropout(0.15))
model_4.add(Dense(num_classes, activation="softmax"))
model_4.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
256
)                 │          
10,496
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (
Dropout
)                    │ (
None
, 
256
)                 │               
0
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
128
)                 │          
32,896
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼──

Total params: 
52,298
 (204.29 KB)

Trainable params: 
52,298
 (204.29 KB)

Non-trainable params: 
0
 (0.00 B)

In [125]:
optimizer = keras.optimizers.RMSprop(learning_rate=0.0005)
model_4.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [126]:
start = time.time()
history = model_4.fit(X_train, y_train, validation_data=(X_val,y_val), batch_size=batch_size, epochs=epochs)
end = time.time()
print("Training time (s):", round(end - start,2))


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9688 - loss: 0.1373 - val_accuracy: 0.9910 - val_loss: 0.0480
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9860 - loss: 0.0664 - val_accuracy: 0.9915 - val_loss: 0.0496
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9882 - loss: 0.0556 - val_accuracy: 0.9923 - val_loss: 0.0429
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9896 - loss: 0.0525 - val_accuracy: 0.9908 - val_loss: 0.0457
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9901 - loss: 0.0514 - val_accuracy: 0.9918 - val_loss: 0.0441
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9900 - loss: 0.0487 - val_accuracy: 0.9930 - val_loss: 0.0423
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9905 - loss: 0.0485 - val_accuracy: 0.9927 - val_loss: 0.0430
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - ac

In [127]:
plot(history,'loss')
plot(history,'accuracy')


In [128]:
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end - start, 2)


results.loc[len(results)] = [
    5,                                  
    [256, 128, 64],                     
    ["selu", "selu", "selu"],            
    epochs,                              
    batch_size,                          
    "rmsprop",                           
    [0.0005, "-"],                       
    "glorot_uniform",                    
    "-",                                  
    train_loss,                          
    val_loss,                          
    train_acc,                         
    val_acc,                             
    train_time                           
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52






3


4


[128, 64, 32]


[elu, elu, elu]


50


32


adam


[0.001, -]


he_normal


-


0.028050


0.054394


0.993813


0.99300


120.34






4


5


[256, 128, 64]


[selu, selu, selu]


50


32


rmsprop


[0.0005, -]


glorot_uniform


-


0.036809


0.043030


0.993688


0.99175


125.55

#### Model 5 — RMSProp

RMSProp adapts learning rates per parameter, which helps when gradients vary significantly. Performance is solid and in line with other models, but does not surpass Adam-based configurations in terms of stability or accuracy.

## Model 5

In [129]:
# use LeCun normal initializer, Adam with amsgrad, activations = relu/tanh
tf.keras.backend.clear_session()
model_5 = Sequential()
model_5.add(Input(shape=(X_train.shape[1],)))
model_5.add(Dense(128, activation="relu", kernel_initializer="lecun_normal"))
model_5.add(Dropout(0.2))
model_5.add(Dense(64, activation="tanh", kernel_initializer="lecun_normal"))
model_5.add(Dense(32, activation="tanh", kernel_initializer="lecun_normal"))  # extra layer
model_5.add(Dropout(0.2))
model_5.add(Dense(num_classes, activation="softmax"))
model_5.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (
Dropout
)                    │ (
None
, 
128
)                 │               
0
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
32
)                  │           
2,080
 │
├──────────────────────────────────────┼─────────────────────────────┼──

Total params: 
15,914
 (62.16 KB)

Trainable params: 
15,914
 (62.16 KB)

Non-trainable params: 
0
 (0.00 B)

In [130]:
optimizer = keras.optimizers.Adam(learning_rate=0.001, amsgrad=True)
model_5.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [131]:
start = time.time()
history = model_5.fit(X_train, y_train, validation_data=(X_val,y_val), batch_size=batch_size, epochs=epochs)
end = time.time()
print("Training time (s):", round(end - start,2))


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9661 - loss: 0.1341 - val_accuracy: 0.9837 - val_loss: 0.0615
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9851 - loss: 0.0631 - val_accuracy: 0.9905 - val_loss: 0.0423
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9884 - loss: 0.0538 - val_accuracy: 0.9920 - val_loss: 0.0406
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9896 - loss: 0.0495 - val_accuracy: 0.9925 - val_loss: 0.0410
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9905 - loss: 0.0462 - val_accuracy: 0.9908 - val_loss: 0.0413
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9901 - loss: 0.0446 - val_accuracy: 0.9927 - val_loss: 0.0375
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9914 - loss: 0.0420 - val_accuracy: 0.9920 - val_loss: 0.0400
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - ac

In [132]:
plot(history,'loss')
plot(history,'accuracy')


In [133]:
# extract metrics for Model 5
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end - start, 2)

# append model summary row into results
results.loc[len(results)] = [
    6,                                
    [128, 64, 32],                  
    ["relu", "tanh", "tanh"],        
    epochs,                            
    batch_size,                        
    "adam_amsgrad",                   
    [0.001, "-"],                  
    "lecun_normal",                    
    "-",                               
    train_loss,                       
    val_loss,                          
    train_acc,                        
    val_acc,                           
    train_time                         
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52






3


4


[128, 64, 32]


[elu, elu, elu]


50


32


adam


[0.001, -]


he_normal


-


0.028050


0.054394


0.993813


0.99300


120.34






4


5


[256, 128, 64]


[selu, selu, selu]


50


32


rmsprop


[0.0005, -]


glorot_uniform


-


0.036809


0.043030


0.993688


0.99175


125.55






5


6


[128, 64, 32]


[relu, tanh, tanh]


50


32


adam_amsgrad


[0.001, -]


lecun_normal


-


0.030109


0.0412

#### Model 6 — Adam AMSGrad

AMSGrad enforces more stable updates by preventing the adaptive learning rate from increasing over time. This model trains smoothly and achieves accuracy comparable to Adam, though without noticeable improvements given the simplicity of the dataset.

## Model 6

In [134]:
# add dropout between layers, use SGD with higher lr and momentum
tf.keras.backend.clear_session()
model_6 = Sequential()
model_6.add(Input(shape=(X_train.shape[1],)))
model_6.add(Dense(512, activation="relu", kernel_initializer="glorot_uniform"))
model_6.add(Dropout(0.3))
model_6.add(Dense(256, activation="relu", kernel_initializer="glorot_uniform"))
model_6.add(Dense(128, activation="relu", kernel_initializer="glorot_uniform"))  # extra layer
model_6.add(Dropout(0.3))
model_6.add(Dense(num_classes, activation="softmax"))
model_6.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
512
)                 │          
20,992
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (
Dropout
)                    │ (
None
, 
512
)                 │               
0
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
256
)                 │         
131,328
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
128
)                 │          
32,896
 │
├──────────────────────────────────────┼─────────────────────────────┼──

Total params: 
186,506
 (728.54 KB)

Trainable params: 
186,506
 (728.54 KB)

Non-trainable params: 
0
 (0.00 B)

In [135]:
optimizer = keras.optimizers.SGD(learning_rate=0.005, momentum=0.8)
model_6.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [136]:
start = time.time()
history = model_6.fit(X_train, y_train, validation_data=(X_val,y_val), batch_size=batch_size, epochs=epochs)
end = time.time()
print("Training time (s):", round(end - start,2))


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9681 - loss: 0.1301 - val_accuracy: 0.9812 - val_loss: 0.0774
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9818 - loss: 0.0809 - val_accuracy: 0.9898 - val_loss: 0.0578
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9853 - loss: 0.0698 - val_accuracy: 0.9918 - val_loss: 0.0532
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9875 - loss: 0.0624 - val_accuracy: 0.9910 - val_loss: 0.0522
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 5ms/step - accuracy: 0.9881 - loss: 0.0563 - val_accuracy: 0.9908 - val_loss: 0.0490
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9884 - loss: 0.0562 - val_accuracy: 0.9915 - val_loss: 0.0472
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 5ms/step - accuracy: 0.9887 - loss: 0.0537 - val_accuracy: 0.9920 - val_loss: 0.0462
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 4ms/step - ac

In [137]:
plot(history,'loss')
plot(history,'accuracy')


In [138]:
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end - start, 2)

# append Model 6 row into results
results.loc[len(results)] = [
    7,                                  
    [512, 256, 128],                    
    ["relu", "relu", "relu"],          
    epochs,                             
    batch_size,                         
    "sgd",                             
    [0.005, 0.8],                       
    "glorot_uniform",                  
    "-",                                 
    train_loss,                       
    val_loss,                          
    train_acc,                          
    val_acc,                            
    train_time                       
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52






3


4


[128, 64, 32]


[elu, elu, elu]


50


32


adam


[0.001, -]


he_normal


-


0.028050


0.054394


0.993813


0.99300


120.34






4


5


[256, 128, 64]


[selu, selu, selu]


50


32


rmsprop


[0.0005, -]


glorot_uniform


-


0.036809


0.043030


0.993688


0.99175


125.55






5


6


[128, 64, 32]


[relu, tanh, tanh]


50


32


adam_amsgrad


[0.001, -]


lecun_normal


-


0.030109


0.0412

#### Model 7 — SGD (Wider Network: 512, 256, 128)

Increasing the network width significantly increases model capacity, but provides no measurable performance benefits. Training time increases, but accuracy and loss remain the same as smaller models, indicating diminishing returns from added complexity.

## Model 7

In [139]:
# change initializer to he_uniform, optimizer Adam with lower lr, activation = relu/tanh
tf.keras.backend.clear_session()
model_7 = Sequential()
model_7.add(Input(shape=(X_train.shape[1],)))
model_7.add(Dense(128, activation="relu", kernel_initializer="he_uniform"))
model_7.add(Dense(64, activation="tanh", kernel_initializer="he_uniform"))
model_7.add(Dense(num_classes, activation="softmax"))
model_7.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃
 Layer (type)                         
┃
 Output Shape                
┃
         Param # 
┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (
Dense
)                        │ (
None
, 
128
)                 │           
5,248
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (
Dense
)                      │ (
None
, 
64
)                  │           
8,256
 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (
Dense
)                      │ (
None
, 
10
)                  │             
650
 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

Total params: 
14,154
 (55.29 KB)

Trainable params: 
14,154
 (55.29 KB)

Non-trainable params: 
0
 (0.00 B)

In [140]:
optimizer = keras.optimizers.Adam(learning_rate=0.0005)
model_7.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])


In [141]:
start = time.time()
history = model_7.fit(X_train, y_train, validation_data=(X_val,y_val), batch_size=batch_size, epochs=epochs)
end = time.time()
print("Training time (s):", round(end - start,2))


Epoch 1/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9516 - loss: 0.1652 - val_accuracy: 0.9843 - val_loss: 0.0671
Epoch 2/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
2s
 3ms/step - accuracy: 0.9864 - loss: 0.0600 - val_accuracy: 0.9893 - val_loss: 0.0502
Epoch 3/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9897 - loss: 0.0487 - val_accuracy: 0.9908 - val_loss: 0.0468
Epoch 4/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9912 - loss: 0.0435 - val_accuracy: 0.9920 - val_loss: 0.0430
Epoch 5/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9916 - loss: 0.0402 - val_accuracy: 0.9912 - val_loss: 0.0430
Epoch 6/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
1s
 3ms/step - accuracy: 0.9921 - loss: 0.0375 - val_accuracy: 0.9908 - val_loss: 0.0427
Epoch 7/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - accuracy: 0.9928 - loss: 0.0356 - val_accuracy: 0.9912 - val_loss: 0.0431
Epoch 8/50

500/500
 
━━━━━━━━━━━━━━━━━━━━
 
3s
 3ms/step - ac

In [142]:
plot(history,'loss')
plot(history,'accuracy')


In [143]:
train_loss = history.history["loss"][-1]
val_loss   = history.history["val_loss"][-1]
train_acc  = history.history["accuracy"][-1]
val_acc    = history.history["val_accuracy"][-1]
train_time = round(end-start, 2)

results.loc[len(results)] = [
    8,
    [128,64],
    ["relu","tanh"],
    epochs,
    batch_size,
    "adam",
    [0.0005, "-"],
    "he_uniform",
    "-",
    train_loss, val_loss, train_acc, val_acc, train_time
]
results


id


hidden


activations


epochs


batch_size


optimizer


lrs


initializer


notes


train_loss


val_loss


train_acc


val_acc


train_time










0


1


[128, 64]


[relu, tanh]


50


32


sgd


[0.001, -]


xavier


-


0.028809


0.039620


0.994250


0.99250


85.13






1


2


[128, 64]


[relu, tanh]


50


32


sgd with momentum


[0.001, 0.9]


xavier


-


0.010314


0.058979


0.997000


0.99125


109.02






2


3


[128, 64]


[relu, tanh]


50


32


adam


[0.001, -]


glorot_uniform


-


0.025173


0.041739


0.994687


0.99300


101.52






3


4


[128, 64, 32]


[elu, elu, elu]


50


32


adam


[0.001, -]


he_normal


-


0.028050


0.054394


0.993813


0.99300


120.34






4


5


[256, 128, 64]


[selu, selu, selu]


50


32


rmsprop


[0.0005, -]


glorot_uniform


-


0.036809


0.043030


0.993688


0.99175


125.55






5


6


[128, 64, 32]


[relu, tanh, tanh]


50


32


adam_amsgrad


[0.001, -]


lecun_normal


-


0.030109


0.0412

#### Graph Comparison

Models using Adam and AMSGrad tend to show smoother and more stable validation loss curves compared to SGD-based models, indicating better generalization and reduced sensitivity to noise. Models with SGD (with or without momentum) converge well but show slightly more fluctuation in validation loss, especially in deeper or wider architectures. Accuracy curves reveal that all models reach very high performance (>99%), though adaptive optimizers like Adam achieve stable peak accuracy earlier in training. Wider and deeper models did not provide meaningful gains and sometimes showed minor signs of overfitting, where training accuracy kept increasing but validation accuracy plateaued. Overall, the curves suggest that while all models perform strongly, Adam-based models maintain the most consistent balance between low validation loss and high validation accuracy, making them the most reliable for deployment.

#### Model 8 — Adam (Low Learning Rate, He Uniform)

This model offers very stable convergence with a low learning rate, maintaining excellent accuracy with minimal fluctuations. It generalizes well and behaves consistently across training and validation.

### Decision

Even though all the models have nearly identical accuracy (≈ 99.9% train, 99.17% validation), the most stable and generalizable performance is observed in Model 8 (Adam with low learning rate and He Uniform initialization). It achieves consistently low loss, smooth convergence, and high validation accuracy without unnecessary model complexity. Adaptive optimizers like Adam handle learning rate adjustments internally, making the model easier to train and more robust.

But, if simplicity and computational efficiency are among the priorities, Then I would choose Model 1 (SGD), as it offers nearly identical performance with fewer parameters and simpler optimization.

### Insight/Recommendation

Deploy Predictive Maintenance Model: Use the trained classifier to identify high-risk turbines early, reducing costly failures and enabling timely repairs.

Use Cost-Sensitive Threshold: Lower the probability threshold (~0.3–0.4) to catch more failures, minimizing false negatives and preventing breakdowns.

Prioritize Key Sensors: Identify the most important sensor features and focus monitoring and maintenance on components that strongly influence failure risk.

Automated Early-Warning System: Connect real-time sensor data to the model to trigger alerts and schedule inspections automatically, improving turbine uptime.

Continuous Retraining: Retrain the model every 3–6 months and monitor FN/FP trends to maintain accuracy as turbine conditions evolve.

Track Cost-Saving KPIs: Measure reductions in replacements, increases in preventive repairs, and improvements in turbine uptime to quantify business value.

### Final Conclusion

ReneWind should deploy the predictive maintenance model to proactively identify high-risk turbines and reduce costly failures. By using a lower, cost-sensitive threshold, the system can minimize false negatives and prevent unexpected breakdowns. Focusing on the most influential sensor features will help optimize maintenance efforts, while integrating real-time model alerts into an automated early-warning workflow will further improve turbine uptime. The model should be retrained every few months to adapt to evolving turbine conditions, and key performance indicators:such as reductions in replacements, increases in preventive repairs, and improved uptime, should be tracked to quantify the business impact.